# SheCraft Jewelry Retrieval Experiments

Compare retrieval methods using notebook 4's selected Gemini embedding model. Answer-keyword evidence is the primary metric; expected-PDF relevance remains a secondary metric. Existing embedding caches are reused when available.

## 1. Setup and paths

In [1]:
import gc
import importlib
import json
import sys
import time
import uuid
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from IPython.display import display
cwd = Path.cwd().resolve()
root_candidates = [cwd, *cwd.parents, cwd / "SheCraft" / "ai-service", cwd / "ai-service"]
PROJECT_ROOT = next(
    (
        candidate
        for candidate in root_candidates
        if (candidate / "src" / "rag" / "scripts").is_dir()
        and (candidate / "rag_documents").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not locate ai-service. Open this notebook from the Jewelry "
        "project or ai-service folder."
    )
RAG_SRC_DIR = PROJECT_ROOT / "src" / "rag"
if str(RAG_SRC_DIR) not in sys.path:
    sys.path.insert(0, str(RAG_SRC_DIR))
DOCUMENTS_DIR = PROJECT_ROOT / "rag_documents"
ARTIFACT_ROOT = PROJECT_ROOT / "data" / "rag" / "processed_documents"
EMBEDDING_ROOT = PROJECT_ROOT / "data" / "rag" / "embeddings"
OUTPUT_DIR = PROJECT_ROOT / "data" / "rag" / "evaluations"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PDF_FILES = sorted(path.resolve() for path in DOCUMENTS_DIR.rglob("*.pdf"))
if not PDF_FILES:
    raise FileNotFoundError(f"No PDFs found under {DOCUMENTS_DIR}")
embedding_module = importlib.reload(importlib.import_module("scripts.5_embeddings"))
retrieval = importlib.reload(importlib.import_module("scripts.7_retrieval"))
evaluation_metrics = importlib.reload(importlib.import_module("evaluations.retrieval_metrics"))
comparison_file = OUTPUT_DIR / "embedding_model_comparison_results.json"
if not comparison_file.is_file(): raise FileNotFoundError("Run notebook 4 first to select an embedding model.")
comparison = json.loads(comparison_file.read_text(encoding="utf-8"))
MODEL_NAME = comparison.get("recommended_model_name")
if MODEL_NAME not in {"gemini-embedding-001", "gemini-embedding-2"}: raise ValueError(f"Unsupported selected model: {MODEL_NAME!r}")
GEMINI_CLIENT = embedding_module.create_gemini_client(PROJECT_ROOT / ".env")
CHUNK_ORDER_MODEL = "BAAI/bge-m3"
RERANKER_NAME = retrieval.DEFAULT_RERANKER_MODEL
DEVICE = "cuda" if torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory >= 4 * 1024**3 else "cpu"
TOP_K = 5
CANDIDATE_K = 30
RUN_RERANKED_METHODS = False
print("PDFs:", len(PDF_FILES))
print("Embedding model:", MODEL_NAME)
print("Reranker:", RERANKER_NAME)
print("Device:", DEVICE)
print("Reranked methods enabled:", RUN_RERANKED_METHODS)


PDFs: 39
Embedding model: gemini-embedding-2
Reranker: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Device: cpu
Reranked methods enabled: False


## 2. Load saved jewelry chunks

In [2]:
chunk_records = []
for chunk_file in sorted(ARTIFACT_ROOT.glob("*/chunks.json")):
    cached_document = json.loads(chunk_file.read_text(encoding="utf-8"))
    chunk_records.extend(cached_document.get("chunks", []))
if not chunk_records:
    raise FileNotFoundError(
        f"No saved chunks found under {ARTIFACT_ROOT}. Run the chunking notebook first."
    )
chunks_by_id = pd.DataFrame(chunk_records).drop_duplicates("chunk_id").set_index("chunk_id")
embedding_artifact_dir = EMBEDDING_ROOT / CHUNK_ORDER_MODEL.replace("/", "__")
manifest_file = embedding_artifact_dir / "manifest.json"
chunk_ids_file = embedding_artifact_dir / "chunks.json"
embeddings_file = embedding_artifact_dir / "embeddings.npy"
for required_file in (manifest_file, chunk_ids_file, embeddings_file):
    if not required_file.is_file():
        raise FileNotFoundError(
            f"Saved retrieval artifact is missing: {required_file}. Run the embedding notebook first."
        )
embedding_manifest = json.loads(manifest_file.read_text(encoding="utf-8"))
if embedding_manifest.get("model_name") != CHUNK_ORDER_MODEL:
    raise ValueError("Chunk-order manifest is not from the configured ordering model.")
embedding_chunk_ids = json.loads(chunk_ids_file.read_text(encoding="utf-8"))["chunk_ids"]
missing_ids = sorted(set(embedding_chunk_ids) - set(chunks_by_id.index))
if missing_ids:
    raise ValueError(f"Embedding cache refers to missing chunk IDs: {missing_ids[:5]}")
chunks_df = chunks_by_id.loc[embedding_chunk_ids].reset_index()
if chunks_df.empty:
    raise ValueError("No chunks were loaded from the saved artifacts.")
if chunks_df["chunk_id"].duplicated().any():
    raise ValueError("Chunk IDs must be unique before retrieval.")
print("Documents:", chunks_df["file_name"].nunique())
print("Chunks:", len(chunks_df))
print("Loaded saved chunk artifacts from:", ARTIFACT_ROOT)
display(chunks_df.groupby("file_name").size().rename("chunks").to_frame())


Documents: 39
Chunks: 283
Loaded saved chunk artifacts from: C:\Users\User\Desktop\LLMs - Samer Saab\Project\Jewelry\SheCraft\ai-service\data\rag\processed_documents


,chunks
file_name,
Ring Size Guide.pdf,20
alexandrite.pdf,4
amber.pdf,4
amethyst.pdf,4
ametrine.pdf,3
aquamarine.pdf,4
birthstone_guide.pdf,10
citrine.pdf,4
diamond.pdf,4


## 3. Load or reuse selected Gemini chunk embeddings

In [3]:
chunk_embedding_result = embedding_module.get_or_create_gemini_embeddings(
    chunks_df["text"].astype(str).tolist(), client=GEMINI_CLIENT,
    text_type="passage", model_name=MODEL_NAME, cache_root=EMBEDDING_ROOT,
    batch_size=100 if MODEL_NAME == "gemini-embedding-001" else 1,
)
embedding_matrix = chunk_embedding_result["embeddings"]
if embedding_matrix.ndim != 2 or embedding_matrix.shape[0] != len(chunks_df): raise ValueError("Gemini embeddings are not aligned with chunks.")
tfidf_index = retrieval.build_tfidf_index(chunks_df)
print("Embedding shape:", embedding_matrix.shape)
print("Chunk embedding cache hit:", chunk_embedding_result["cache_hit"])
print("TF-IDF matrix:", tfidf_index["matrix"].shape)

Embedding shape: (283, 3072)
Chunk embedding cache hit: True
TF-IDF matrix: (283, 19796)


## 4. Build an in-memory Qdrant index for Qdrant retrieval methods

In [4]:
from qdrant_client import QdrantClient, models
COLLECTION_NAME = "shecraft_jewelry_retrieval_test"
qdrant_client = QdrantClient(":memory:")
qdrant_client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(
        size=int(embedding_matrix.shape[1]),
        distance=models.Distance.COSINE,
    ),
)
def to_json_value(value):
    if isinstance(value, np.generic):
        return value.item()
    if pd.isna(value):
        return None
    return value
points = []
for position, row in chunks_df.iterrows():
    payload = {str(key): to_json_value(value) for key, value in row.items()}
    point_id = str(uuid.uuid5(uuid.NAMESPACE_URL, str(row["chunk_id"])))
    points.append(models.PointStruct(
        id=point_id,
        vector=embedding_matrix[position].astype(float).tolist(),
        payload=payload,
    ))
    if len(points) >= 128:
        qdrant_client.upsert(collection_name=COLLECTION_NAME, points=points)
        points = []
if points:
    qdrant_client.upsert(collection_name=COLLECTION_NAME, points=points)
qdrant_count = qdrant_client.count(COLLECTION_NAME, exact=True).count
if qdrant_count != len(chunks_df):
    raise ValueError(f"Qdrant points ({qdrant_count}) != chunks ({len(chunks_df)}).")
print("In-memory Qdrant points:", qdrant_count)


In-memory Qdrant points: 283


## 5. Load the cross-encoder reranker when requested

In [5]:
reranker = None
if RUN_RERANKED_METHODS:
    from sentence_transformers import CrossEncoder
    model_kwargs = {"torch_dtype": torch.float16} if DEVICE == "cuda" else {}
    reranker = CrossEncoder(
        RERANKER_NAME,
        device=DEVICE,
        local_files_only=True,
        model_kwargs=model_kwargs,
    )
    print("Reranker ready:", RERANKER_NAME)
else:
    print("Reranked methods are skipped. Set RUN_RERANKED_METHODS = True to include them.")


Reranked methods are skipped. Set RUN_RERANKED_METHODS = True to include them.


## 6. Jewelry test questions

Cases and expected source files are stored in `data/rag/evaluations/retrieval_method_benchmark_100.json`. Edit that file to add or change questions. Multi-query retrieval uses the supplied paraphrases and does not call an external API.

In [6]:
TEST_CASES_FILE = OUTPUT_DIR / "retrieval_method_benchmark_100.json"
TEST_CASES = json.loads(TEST_CASES_FILE.read_text(encoding="utf-8"))
if len(TEST_CASES) != 100: raise ValueError(f"Expected 100 test cases; found {len(TEST_CASES)}")
if any(not c.get("expected_keywords") for c in TEST_CASES): raise ValueError("Every test case must include expected_keywords.")
available_files = set(chunks_df["file_name"].astype(str))
missing_targets = sorted({f for c in TEST_CASES for f in c.get("expected_documents", [])} - available_files)
if missing_targets: raise ValueError(f"Test cases target files absent from corpus: {missing_targets}")
def normalize_keyword_text(value):
    import unicodedata
    normalized = unicodedata.normalize("NFKD", str(value).casefold())
    return "".join(ch for ch in normalized if ch.isalnum())
query_texts = list(dict.fromkeys(t.strip() for c in TEST_CASES for t in [c["query"], *c.get("variants", [])] if t and t.strip()))
query_embedding_result = embedding_module.get_or_create_gemini_embeddings(query_texts, client=GEMINI_CLIENT, text_type="query", model_name=MODEL_NAME, cache_root=EMBEDDING_ROOT, batch_size=100 if MODEL_NAME == "gemini-embedding-001" else 1)
query_vectors = dict(zip(query_texts, query_embedding_result["embeddings"]))
class CachedGeminiQueryModel:
    def encode(self, sentences, **kwargs):
        vectors=[]
        for sentence in sentences:
            text=str(sentence).strip()
            if text not in query_vectors:
                result=embedding_module.get_or_create_gemini_embeddings([text],client=GEMINI_CLIENT,text_type="query",model_name=MODEL_NAME,cache_root=EMBEDDING_ROOT,batch_size=1)
                query_vectors[text]=result["embeddings"][0]
            vectors.append(query_vectors[text])
        return np.asarray(vectors,dtype=np.float32)
embedding_model=CachedGeminiQueryModel()
print("Test cases:",len(TEST_CASES))
print("Languages:",pd.Series([c["language"] for c in TEST_CASES]).value_counts().to_dict())
print("Cross-PDF cases:",sum(c.get("cross_pdf",False) for c in TEST_CASES))
print("Query embedding cache hit:",query_embedding_result["cache_hit"])

Test cases: 100
Languages: {'en': 60, 'ar': 20, 'fr': 20}
Cross-PDF cases: 19
Query embedding cache hit: True


## 7. Run available retrieval methods on the same test cases

In [7]:
methods = {
    "dense": lambda query, case: retrieval.retrieve_dense(
        query, embedding_model, embedding_matrix, chunks_df,
        top_k=TOP_K, model_name=MODEL_NAME,
    ),
    "dense_qdrant": lambda query, case: retrieval.retrieve_dense_qdrant(
        query, qdrant_client, embedding_model,
        collection_name=COLLECTION_NAME, top_k=TOP_K, model_name=MODEL_NAME,
    ),
    "tfidf": lambda query, case: retrieval.retrieve_tfidf(
        query, tfidf_index, chunks_df, top_k=TOP_K,
    ),
    "hybrid_rrf": lambda query, case: retrieval.retrieve_hybrid(
        query, embedding_model, embedding_matrix, chunks_df, tfidf_index,
        top_k=TOP_K, candidate_k=CANDIDATE_K, model_name=MODEL_NAME,
    ),
    "hybrid_rrf_qdrant": lambda query, case: retrieval.retrieve_hybrid_qdrant(
        query, qdrant_client, embedding_model, chunks_df, tfidf_index,
        collection_name=COLLECTION_NAME, top_k=TOP_K,
        candidate_k=CANDIDATE_K, model_name=MODEL_NAME,
    ),
    "multi_query": lambda query, case: retrieval.retrieve_multi_query(
        [query, *case.get("variants", [])], embedding_model,
        embedding_matrix, chunks_df, top_k=TOP_K,
        candidate_k=CANDIDATE_K, model_name=MODEL_NAME,
    ),
}
if RUN_RERANKED_METHODS:
    methods.update({
        "dense_reranked": lambda query, case: retrieval.retrieve_dense_reranked(
            query, embedding_model, embedding_matrix, chunks_df, reranker,
            top_k=TOP_K, candidate_k=CANDIDATE_K, model_name=MODEL_NAME,
        ),
        "dense_qdrant_reranked": lambda query, case: retrieval.retrieve_dense_qdrant_reranked(
            query, qdrant_client, embedding_model, reranker,
            collection_name=COLLECTION_NAME, top_k=TOP_K,
            candidate_k=CANDIDATE_K, model_name=MODEL_NAME,
        ),
        "hybrid_reranked": lambda query, case: retrieval.retrieve_hybrid_reranked(
            query, embedding_model, embedding_matrix, chunks_df, tfidf_index, reranker,
            top_k=TOP_K, candidate_k=CANDIDATE_K, rerank_k=CANDIDATE_K,
            model_name=MODEL_NAME,
        ),
        "hybrid_reranked_qdrant": lambda query, case: retrieval.retrieve_hybrid_reranked_qdrant(
            query, qdrant_client, embedding_model, chunks_df, tfidf_index, reranker,
            collection_name=COLLECTION_NAME, top_k=TOP_K,
            candidate_k=CANDIDATE_K, rerank_k=CANDIDATE_K,
            model_name=MODEL_NAME,
        ),
    })
detail_rows = []
case_rows = []
for method_name, retrieve_fn in methods.items():
    print("Running:", method_name)
    for case in TEST_CASES:
        started = time.perf_counter()
        results = retrieve_fn(case["query"], case)
        latency_ms = (time.perf_counter() - started) * 1000
        relevant_rank = None
        answer_relevant_rank = None
        expected_documents = set(case.get("expected_documents", []))
        expected_keywords = case["expected_keywords"]
        matched_keywords_at_5 = set()
        for _, row in results.iterrows():
            relevant = str(row["file_name"]) in expected_documents
            normalized_text = normalize_keyword_text(row["text"])
            matched_keywords = [kw for kw in expected_keywords if normalize_keyword_text(kw) in normalized_text]
            answer_relevant = relevant and bool(matched_keywords)
            if answer_relevant and answer_relevant_rank is None: answer_relevant_rank = int(row["rank"])
            if relevant: matched_keywords_at_5.update(matched_keywords)
            if relevant and relevant_rank is None:
                relevant_rank = int(row["rank"])
            detail_rows.append({
                "method": method_name,
                "case_id": case["id"],
                "query": case["query"],
                "expected_documents": case.get("expected_documents", []),
                "rank": int(row["rank"]),
                "score": float(row["score"]),
                "chunk_id": str(row["chunk_id"]),
                "file_name": str(row["file_name"]),
                "page_number": int(row["page_number"]),
                "text": str(row["text"]),
                "relevant": bool(relevant),
                "answer_relevant": bool(answer_relevant),
                "matched_keywords": matched_keywords,
            })
        case_rows.append({
            "method": method_name,
            "case_id": case["id"],
            "query": case["query"],
            "expected_documents": case.get("expected_documents", []),
            "relevant_rank": relevant_rank,
            "answer_relevant_rank": answer_relevant_rank,
            "expected_keywords": expected_keywords,
            "matched_keywords_at_5": sorted(matched_keywords_at_5),
            "keyword_coverage_at_5": len(matched_keywords_at_5) / len(expected_keywords) if expected_keywords else 0.0,
            "latency_ms": latency_ms,
        })
case_results = pd.DataFrame(case_rows)
detailed_results = pd.DataFrame(detail_rows)
print("Completed:", len(case_results), "method/query runs")


Running: dense
Running: dense_qdrant
Running: tfidf
Running: hybrid_rrf
Running: hybrid_rrf_qdrant
Running: multi_query
Completed: 600 method/query runs


## 8. Compare method quality and speed

In [8]:
metrics, metrics_by_language, cross_document_metrics, metrics_by_document = evaluation_metrics.summarize_retrieval_metrics(TEST_CASES, case_results, detailed_results, cutoffs=(1,3,5))
def answer_metric_table(frame, groups=()):
    grouped=frame.groupby(list(groups),sort=False) if groups else [(None,frame)]; rows=[]
    for keys,g in grouped:
        row={}
        if groups:
            vals=keys if isinstance(keys,tuple) else (keys,); row.update(dict(zip(groups,vals)))
        ranks=pd.to_numeric(g["answer_relevant_rank"],errors="coerce")
        row["queries"]=len(g)
        for k in (1,3,5): row[f"AnswerRecall@{k}"]=ranks.le(k).fillna(False).mean()
        row["AnswerMRR@5"]=(1.0/ranks.where(ranks.le(5))).fillna(0).mean()
        row["KeywordCoverage@5"]=g["keyword_coverage_at_5"].mean(); rows.append(row)
    return pd.DataFrame(rows)
answer_metrics=answer_metric_table(case_results,("method",))
langs={c["id"]:c["language"] for c in TEST_CASES}
answer_metrics_by_language=answer_metric_table(case_results.assign(language=case_results["case_id"].map(langs)),("method","language"))
cross_ids={c["id"] for c in TEST_CASES if c.get("cross_pdf",False)}
answer_cross_document_metrics=answer_metric_table(case_results[case_results["case_id"].isin(cross_ids)],("method",))
print("PDF relevance metrics (secondary)"); display(metrics.round(4)); display(metrics_by_language.round(4)); display(cross_document_metrics.round(4))
print("Answer keyword metrics (primary)"); display(answer_metrics.round(4)); display(answer_metrics_by_language.round(4)); display(answer_cross_document_metrics.round(4))
display(metrics_by_document.round(4))
misses=case_results[case_results["answer_relevant_rank"].isna()].sort_values(["method","case_id"])
display(misses if len(misses) else pd.DataFrame({"result":["No answer-bearing misses"]}))

PDF relevance metrics (secondary)


,method,queries,Recall@1,TargetCoverage@1,Recall@3,TargetCoverage@3,Recall@5,TargetCoverage@5,MRR@5,mean_latency_ms
0,dense,100,0.95,0.8583,0.99,0.9600,0.99,0.9700,0.9667,4.7581
1,multi_query,100,0.95,0.8583,0.99,0.9600,0.99,0.9700,0.9667,8.2002
2,dense_qdrant,100,0.95,0.8583,0.99,0.9600,0.99,0.9700,0.9667,10.1099
3,hybrid_rrf,100,0.91,0.8183,0.98,0.9450,0.98,0.9700,0.9400,9.4885
4,hybrid_rrf_qdrant,100,0.91,0.8183,0.98,0.9450,0.98,0.9700,0.9400,13.7902
5,tfidf,100,0.58,0.4883,0.65,0.6067,0.68,0.6467,0.6170,1.5181


,method,language,queries,Recall@1,Recall@5,mean_latency_ms
0,dense,en,60,0.9667,1.0000,4.7997
1,dense,ar,20,0.9000,0.9500,4.0224
2,dense,fr,20,0.9500,1.0000,5.3688
3,dense_qdrant,en,60,0.9667,1.0000,11.3092
4,dense_qdrant,ar,20,0.9000,0.9500,8.1905
5,dense_qdrant,fr,20,0.9500,1.0000,8.4313
6,tfidf,en,60,0.8667,0.9833,1.5827
7,tfidf,ar,20,0.0000,0.0000,1.1088
8,tfidf,fr,20,0.3000,0.4500,1.7340
9,hybrid_rrf,en,60,0.9500,1.0000,10.1540


,method,queries,Recall@1,Recall@5
0,dense,19,0.9474,1.0
1,dense_qdrant,19,0.9474,1.0
2,tfidf,19,0.9474,1.0
3,hybrid_rrf,19,0.9474,1.0
4,hybrid_rrf_qdrant,19,0.9474,1.0
5,multi_query,19,0.9474,1.0


Answer keyword metrics (primary)


,method,queries,AnswerRecall@1,AnswerRecall@3,AnswerRecall@5,AnswerMRR@5,KeywordCoverage@5
0,dense,100,0.85,0.98,0.98,0.9100,0.9285
1,dense_qdrant,100,0.85,0.98,0.98,0.9100,0.9285
2,tfidf,100,0.50,0.62,0.65,0.5582,0.5665
3,hybrid_rrf,100,0.81,0.95,0.98,0.8803,0.9035
4,hybrid_rrf_qdrant,100,0.81,0.95,0.98,0.8803,0.9035
5,multi_query,100,0.85,0.98,0.98,0.9100,0.9285


,method,language,queries,AnswerRecall@1,AnswerRecall@3,AnswerRecall@5,AnswerMRR@5,KeywordCoverage@5
0,dense,en,60,0.8833,0.9833,0.9833,0.9278,0.9342
1,dense,ar,20,0.8500,0.9500,0.9500,0.9000,0.9500
2,dense,fr,20,0.7500,1.0000,1.0000,0.8667,0.8900
3,dense_qdrant,en,60,0.8833,0.9833,0.9833,0.9278,0.9342
4,dense_qdrant,ar,20,0.8500,0.9500,0.9500,0.9000,0.9500
5,dense_qdrant,fr,20,0.7500,1.0000,1.0000,0.8667,0.8900
6,tfidf,en,60,0.7833,0.9333,0.9500,0.8533,0.8275
7,tfidf,ar,20,0.0000,0.0000,0.0000,0.0000,0.0000
8,tfidf,fr,20,0.1500,0.3000,0.4000,0.2308,0.3500
9,hybrid_rrf,en,60,0.8500,1.0000,1.0000,0.9167,0.9375


,method,queries,AnswerRecall@1,AnswerRecall@3,AnswerRecall@5,AnswerMRR@5,KeywordCoverage@5
0,dense,19,0.8421,0.9474,0.9474,0.8860,0.8579
1,dense_qdrant,19,0.8421,0.9474,0.9474,0.8860,0.8579
2,tfidf,19,0.9474,1.0000,1.0000,0.9649,0.8368
3,hybrid_rrf,19,0.8421,1.0000,1.0000,0.9123,0.9211
4,hybrid_rrf_qdrant,19,0.8421,1.0000,1.0000,0.9123,0.9211
5,multi_query,19,0.8421,0.9474,0.9474,0.8860,0.8579


,method,document,queries,Recall@1,Recall@5
0,dense,Ring Size Guide.pdf,5,1.00,1.0
1,dense,alexandrite.pdf,3,1.00,1.0
2,dense,amber.pdf,3,1.00,1.0
3,dense,amethyst.pdf,4,0.75,1.0
4,dense,ametrine.pdf,2,1.00,1.0
...,...,...,...,...,...
229,multi_query,tanzanite.pdf,2,1.00,1.0
230,multi_query,topaz.pdf,3,1.00,1.0
231,multi_query,tourmaline.pdf,3,1.00,1.0
232,multi_query,turquoise.pdf,3,1.00,1.0


,method,case_id,query,expected_documents,relevant_rank,answer_relevant_rank,expected_keywords,matched_keywords_at_5,keyword_coverage_at_5,latency_ms
46,dense,retrieval_047,How can chain-link styles be combined when cre...,"[jewelry_chain_guide.pdf, necklace_stacking_gu...",1.0,NaN,"[link styles, box chains, cable link, layered ...",[],0.0,4.0801
69,dense,retrieval_070,كيف يصف الدليل حجر العقيق من حيث اللون والتركي...,[garnet.pdf],NaN,NaN,"[closely related minerals, almost every color,...",[],0.0,3.1971
146,dense_qdrant,retrieval_047,How can chain-link styles be combined when cre...,"[jewelry_chain_guide.pdf, necklace_stacking_gu...",1.0,NaN,"[link styles, box chains, cable link, layered ...",[],0.0,6.8323
169,dense_qdrant,retrieval_070,كيف يصف الدليل حجر العقيق من حيث اللون والتركي...,[garnet.pdf],NaN,NaN,"[closely related minerals, almost every color,...",[],0.0,7.5064
369,hybrid_rrf,retrieval_070,كيف يصف الدليل حجر العقيق من حيث اللون والتركي...,[garnet.pdf],NaN,NaN,"[closely related minerals, almost every color,...",[],0.0,9.0856
382,hybrid_rrf,retrieval_083,Quelle structure produit les eclats scintillan...,[sunstone.pdf],NaN,NaN,"[aventurescence, flat reflective inclusions, c...",[],0.0,10.5124
469,hybrid_rrf_qdrant,retrieval_070,كيف يصف الدليل حجر العقيق من حيث اللون والتركي...,[garnet.pdf],NaN,NaN,"[closely related minerals, almost every color,...",[],0.0,11.7679
482,hybrid_rrf_qdrant,retrieval_083,Quelle structure produit les eclats scintillan...,[sunstone.pdf],NaN,NaN,"[aventurescence, flat reflective inclusions, c...",[],0.0,14.2406
546,multi_query,retrieval_047,How can chain-link styles be combined when cre...,"[jewelry_chain_guide.pdf, necklace_stacking_gu...",1.0,NaN,"[link styles, box chains, cable link, layered ...",[],0.0,8.9943
569,multi_query,retrieval_070,كيف يصف الدليل حجر العقيق من حيث اللون والتركي...,[garnet.pdf],NaN,NaN,"[closely related minerals, almost every color,...",[],0.0,6.6382


## 9. Inspect one case across all methods

In [9]:
CASE_TO_INSPECT = TEST_CASES[0]["id"]
case = next(item for item in TEST_CASES if item["id"] == CASE_TO_INSPECT)
display(pd.DataFrame([{
    "case_id": case["id"],
    "query": case["query"],
    "expected_documents": case.get("expected_documents", []),
}]))
inspection = detailed_results[detailed_results["case_id"] == CASE_TO_INSPECT].sort_values(
    ["method", "rank"]
)
display(inspection[[
    "method", "rank", "score", "file_name", "page_number", "relevant", "text"
]])


,case_id,query,expected_documents
0,retrieval_001,Should pearl jewelry be cleaned with a steam c...,[pearl_care_cleaning_guide.pdf]


,method,rank,score,file_name,page_number,relevant,text
0,dense,1,0.822276,pearl_care_cleaning_guide.pdf,2,True,Pearls should never be cleaned in an ultrasoni...
1,dense,2,0.722459,pearl_care_cleaning_guide.pdf,2,True,"Pearl’s toughness is usually good, but aging, ..."
2,dense,3,0.719689,pearl_care_cleaning_guide.pdf,1,True,Source: https://www.gia.edu/pearl-care-cleanin...
3,dense,4,0.686893,gemstone_diamond_pearl_buying_guide.pdf,6,False,"pearl’s value generally is based on its size, ..."
4,dense,5,0.680314,engagement_ring_settings_guide.pdf,14,False,Cleaning And Maintenance Regular cleaning help...
500,dense_qdrant,1,0.822276,pearl_care_cleaning_guide.pdf,2,True,Pearls should never be cleaned in an ultrasoni...
501,dense_qdrant,2,0.722459,pearl_care_cleaning_guide.pdf,2,True,"Pearl’s toughness is usually good, but aging, ..."
502,dense_qdrant,3,0.719689,pearl_care_cleaning_guide.pdf,1,True,Source: https://www.gia.edu/pearl-care-cleanin...
503,dense_qdrant,4,0.686893,gemstone_diamond_pearl_buying_guide.pdf,6,False,"pearl’s value generally is based on its size, ..."
504,dense_qdrant,5,0.680314,engagement_ring_settings_guide.pdf,14,False,Cleaning And Maintenance Regular cleaning help...


## 10. Try your own query

In [10]:
CUSTOM_QUERY = "How should I care for pearl jewelry?"
CUSTOM_VARIANTS = [
    "What is the best way to clean and protect pearls?",
    "Which cleaning products or methods can damage pearls?",
]
custom_case = {"id": "custom", "variants": CUSTOM_VARIANTS}
custom_rows = []
for method_name, retrieve_fn in methods.items():
    result = retrieve_fn(CUSTOM_QUERY, custom_case)
    for _, row in result.iterrows():
        custom_rows.append({
            "method": method_name,
            "rank": int(row["rank"]),
            "score": float(row["score"]),
            "file_name": str(row["file_name"]),
            "page_number": int(row["page_number"]),
            "text": str(row["text"]),
        })
display(pd.DataFrame(custom_rows).sort_values(["method", "rank"]))


gemini-embedding-2 checkpoint saved: batch 1/1 (1/1 texts)
gemini-embedding-2 checkpoint saved: batch 1/1 (1/1 texts)
gemini-embedding-2 checkpoint saved: batch 1/1 (1/1 texts)


,method,rank,score,file_name,page_number,text
0,dense,1,0.810236,pearl_care_cleaning_guide.pdf,2,Pearls should never be cleaned in an ultrasoni...
1,dense,2,0.757798,pearl_care_cleaning_guide.pdf,1,Source: https://www.gia.edu/pearl-care-cleanin...
2,dense,3,0.754644,pearl_care_cleaning_guide.pdf,2,"Pearl’s toughness is usually good, but aging, ..."
3,dense,4,0.751113,pearl_care_cleaning_guide.pdf,3,With proper care cultured pearls can provide t...
4,dense,5,0.714813,gemstone_diamond_pearl_buying_guide.pdf,6,"pearl’s value generally is based on its size, ..."
5,dense_qdrant,1,0.810236,pearl_care_cleaning_guide.pdf,2,Pearls should never be cleaned in an ultrasoni...
6,dense_qdrant,2,0.757798,pearl_care_cleaning_guide.pdf,1,Source: https://www.gia.edu/pearl-care-cleanin...
7,dense_qdrant,3,0.754644,pearl_care_cleaning_guide.pdf,2,"Pearl’s toughness is usually good, but aging, ..."
8,dense_qdrant,4,0.751113,pearl_care_cleaning_guide.pdf,3,With proper care cultured pearls can provide t...
9,dense_qdrant,5,0.714813,gemstone_diamond_pearl_buying_guide.pdf,6,"pearl’s value generally is based on its size, ..."


## 11. Save results

In [11]:
RESULTS_FILE = OUTPUT_DIR / "jewelry_retrieval_method_results.json"
payload = {
    "schema_version": 2,
    "embedding_model": MODEL_NAME,
    "reranker_model": RERANKER_NAME if RUN_RERANKED_METHODS else None,
    "reranked_methods_enabled": RUN_RERANKED_METHODS,
    "chunk_count": len(chunks_df),
    "document_count": int(chunks_df["file_name"].nunique()),
    "top_k": TOP_K,
    "candidate_k": CANDIDATE_K,
    "test_case_file": str(TEST_CASES_FILE),
    "test_cases": TEST_CASES,
    "metrics": metrics.to_dict(orient="records"),
    "metrics_by_language": metrics_by_language.to_dict(orient="records"),
    "cross_document_metrics": cross_document_metrics.to_dict(orient="records"),
    "metrics_by_document": metrics_by_document.to_dict(orient="records"),
    "answer_metrics": answer_metrics.to_dict(orient="records"),
    "answer_metrics_by_language": answer_metrics_by_language.to_dict(orient="records"),
    "answer_cross_document_metrics": answer_cross_document_metrics.to_dict(orient="records"),
    "case_results": case_results.where(pd.notna(case_results), None).to_dict(orient="records"),
    "retrieved_results": detailed_results.to_dict(orient="records"),
}
RESULTS_FILE.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", RESULTS_FILE)


Saved: C:\Users\User\Desktop\LLMs - Samer Saab\Project\Jewelry\SheCraft\ai-service\data\rag\evaluations\jewelry_retrieval_method_results.json


## 12. Cleanup

In [12]:
qdrant_client.close()
del embedding_model, embedding_matrix
if reranker is not None:
    del reranker
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Retrieval experiment complete.")


Retrieval experiment complete.
